# Proyecto 1: Entrenamiento de un agente PPO en `CartPole-v1` con Stable-Baselines3
En este notebook entrenamos y evaluamos un agente de **Aprendizaje por Refuerzo** (*Reinforcement Learning*) usando el algoritmo **PPO** (*Proximal Policy Optimization*) de **Stable-Baselines3** sobre el entorno clásico **`CartPole-v1`** de **Gymnasium**.

## Objetivo del problema
El objetivo en `CartPole-v1` es mantener equilibrado un poste vertical sobre un carrito móvil aplicando fuerzas discretas hacia la **izquierda** o la **derecha** (+1 de recompensa por cada paso en el que el poste no cae ni el carrito sale de los límites, hasta un máximo de **500 pasos** por episodio).

## Desarrollo del notebook
1. **Entrenar un modelo base de PPO** sobre `CartPole-v1` y evaluar su rendimiento medio.
2. **Experimentar con hiperparámetros** clave del algoritmo, en concreto:
   - **`learning_rate`**: tamaño del paso al actualizar los pesos de la red neuronal.
   - **`n_steps`**: cantidad de pasos recolectados en el entorno por cada actualización de la política.
3. **Registrar y comparar los resultados** mediante **TensorBoard** (o **Weights & Biases**) para analizar la estabilidad y velocidad de aprendizaje.

In [6]:
import os
import gymnasium as gym
import numpy as np
from matplotlib import animation
from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

print("Gymnasium:", gym.__version__)

Gymnasium: 1.3.0


In [2]:
log_dir = "./tb_logs"
os.makedirs(log_dir, exist_ok=True)
os.makedirs("./models", exist_ok=True)

env = Monitor(gym.make("CartPole-v1"))

model_base = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    verbose=1,
    tensorboard_log=log_dir,
)

model_base.learn(total_timesteps=50_000, tb_log_name="ppo_base_lr3e-4_n2048")
model_base.save("./models/ppo_base")

mean_reward, std_reward = evaluate_policy(model_base, env, n_eval_episodes=10, deterministic=True)
print(f"Recompensa media (10 episodios): {mean_reward:.2f} +/- {std_reward:.2f}")

env.close()

Using cpu device
Wrapping the env in a DummyVecEnv.
Logging to ./tb_logs/ppo_base_lr3e-4_n2048_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.4     |
|    ep_rew_mean     | 21.4     |
| time/              |          |
|    fps             | 1810     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 28.7       |
|    ep_rew_mean          | 28.7       |
| time/                   |            |
|    fps                  | 901        |
|    iterations           | 2          |
|    time_elapsed         | 4          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00832276 |
|    clip_fraction        | 0.0923     |
|    clip_range           | 0.2        |
|    entropy_loss         | -0

In [3]:
experimentos = [
    {"nombre": "lr1e-3_n512",  "learning_rate": 1e-3, "n_steps": 512},
    {"nombre": "lr3e-4_n2048", "learning_rate": 3e-4, "n_steps": 2048},
    {"nombre": "lr1e-4_n4096", "learning_rate": 1e-4, "n_steps": 4096},
]

resultados = []

for exp in experimentos:
    eval_env = gym.make("CartPole-v1")
    train_env = Monitor(gym.make("CartPole-v1"))

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=exp["learning_rate"],
        n_steps=exp["n_steps"],
        batch_size=64,
        verbose=0,
        tensorboard_log=log_dir,
    )

    model.learn(total_timesteps=50_000, tb_log_name=exp["nombre"])
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)

    resultados.append((exp["nombre"], exp["learning_rate"], exp["n_steps"], mean_r, std_r))
    print(f"{exp['nombre']:15s} | lr={exp['learning_rate']} | n_steps={exp['n_steps']} | reward={mean_r:.2f} +/- {std_r:.2f}")

    train_env.close()
    eval_env.close()

/home/sergi/anaconda3/envs/RLenv/lib/python3.10/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


lr1e-3_n512     | lr=0.001 | n_steps=512 | reward=500.00 +/- 0.00
lr3e-4_n2048    | lr=0.0003 | n_steps=2048 | reward=500.00 +/- 0.00
lr1e-4_n4096    | lr=0.0001 | n_steps=4096 | reward=499.50 +/- 1.50


In [4]:
%load_ext tensorboard
%tensorboard --logdir ./tb_logs